# 02 — Qualidade de dados com Great Expectations

> **A pergunta desta etapa:** o dado que acabei de ingerir **merece confiança**?

Repare que esta não é a fase 2. É um **portão** entre as fases. Validar depois de
tudo pronto é tarde demais: o número errado já foi para a apresentação.

## Por que isso é tão importante em dados

Em software, quando algo quebra, o programa para e você vê um *traceback*.
Em dados, quando algo quebra, **o pipeline roda com sucesso e o número fica
errado**. Ninguém é avisado. O erro é descoberto três meses depois, por um
usuário, na reunião errada.

Alguns exemplos que este projeto encontrou de verdade:

* a API trocou o status `"+1 Lap"` por `"Lapped"` — quem tivesse fixado o texto
  antigo passaria a contar **10 abandonos por corrida** em vez de 0, sem nenhum
  erro de execução;
* uma parada de box sob bandeira vermelha durou **543 segundos** e destruiu a
  média de pit stop do GP inteiro;
* um `position` que vem como `"1"` (texto) ordena `"10" < "2"`.

Nenhum desses casos levanta exceção. Todos produzem número errado.

**Great Expectations** existe para transformar essas suposições implícitas em
**regras explícitas, versionadas e executáveis**.

In [ ]:
# --- Preparação do ambiente (rode esta célula primeiro) --------------------
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

print("Raiz do projeto:", RAIZ)
print("Python:", sys.version.split()[0])


## 1. Antes de validar: RAW → BRONZE

Não dá para validar JSON aninhado com regras de tabela. Então o primeiro passo é
achatar o payload em linhas e colunas — e **só isso**.

A camada bronze tem uma única responsabilidade: transformar JSON em tabela.

* ❌ não converte tipo (tudo continua texto, como veio);
* ❌ não filtra nada;
* ❌ não aplica regra de negócio.

Por que essa disciplina? Porque assim **cada célula do bronze pode ser conferida
contra o JSON original**. É essa fidelidade que torna honesta a pergunta "a fonte
continua cumprindo o contrato?".

In [2]:
from f1_pipeline import config
from f1_pipeline.processing.bronze import build_bronze

bronze = build_bronze(config.SEASONS)

print()
for nome, df in bronze.items():
    print(f"{nome:<24} {len(df):>6} linhas x {df.shape[1]:>2} colunas")


22:03:48 | INFO    | f1_pipeline.processing.bronze | BRONZE races                  |     90 linhas | 12 colunas -> races.parquet
22:03:48 | INFO    | f1_pipeline.processing.bronze | BRONZE results                |   1799 linhas | 26 colunas -> results.parquet
22:03:49 | INFO    | f1_pipeline.processing.bronze | BRONZE qualifying             |   1798 linhas | 11 colunas -> qualifying.parquet
22:03:49 | INFO    | f1_pipeline.processing.bronze | BRONZE pit_stops              |   3340 linhas |  9 colunas -> pit_stops.parquet
22:03:49 | INFO    | f1_pipeline.processing.bronze | BRONZE driver_standings       |     89 linhas | 13 colunas -> driver_standings.parquet
22:03:49 | INFO    | f1_pipeline.processing.bronze | BRONZE constructor_standings  |     40 linhas |  9 colunas -> constructor_standings.parquet

races                        90 linhas x 12 colunas
results                    1799 linhas x 26 colunas
qualifying                 1798 linhas x 11 colunas
pit_stops                  3340

In [3]:
resultados_bronze = bronze["results"]
resultados_bronze.head(3)


,season,round,race_name,date,driver_id,driver_code,driver_number,given_name,family_name,driver_nationality,date_of_birth,constructor_id,constructor_name,constructor_nationality,grid,position,position_text,points,laps,status,time_millis,time_text,fastest_lap_rank,fastest_lap_number,fastest_lap_time,fastest_lap_speed_kph
0,2021,1,Bahrain Grand Prix,2021-03-28,hamilton,HAM,44,Lewis,Hamilton,British,1985-01-07,mercedes,Mercedes,German,2,1,1,25,56,Finished,5523897,1:32:03.897,4,44,1:34.015,207.235
1,2021,1,Bahrain Grand Prix,2021-03-28,max_verstappen,VER,33,Max,Verstappen,Dutch,1997-09-30,red_bull,Red Bull,Austrian,1,2,2,18,56,Finished,5524642,+0.745,2,41,1:33.228,208.984
2,2021,1,Bahrain Grand Prix,2021-03-28,bottas,BOT,77,Valtteri,Bottas,Finnish,1989-08-28,mercedes,Mercedes,German,3,3,3,16,56,Finished,5561280,+37.383,1,56,1:32.090,211.566


In [4]:
# A prova de que bronze é "fiel à fonte": TODA coluna ainda é texto.
resultados_bronze.dtypes.value_counts()


object    26
Name: count, dtype: int64

## 2. O vocabulário do Great Expectations

O GX 1.x tem seis peças. Vale decorar a ordem em que elas se encaixam:

| Peça | O que é | No nosso projeto |
|---|---|---|
| **Data Context** | o "projeto" de qualidade | pasta `gx/` na raiz |
| **Data Source** | de onde o dado vem | DataFrames pandas em memória |
| **Data Asset** | uma tabela dentro da fonte | `bronze_results`, `silver_fact_result`... |
| **Batch Definition** | qual recorte validar | a tabela inteira |
| **Expectation Suite** | o conjunto de regras | `bronze_results` (10 regras) |
| **Validation Definition** | amarra "este lote" + "estas regras" | criada automaticamente |
| **Checkpoint** | executa e dispara ações | atualiza os Data Docs ao final |

Uma **Expectation** é uma afirmação sobre o dado, escrita em inglês quase
natural: *"espero que a coluna `driver_id` não tenha nulos"*. O GX tem ~60 delas
prontas.

In [5]:
import great_expectations as gx

print("Great Expectations", gx.__version__)

disponiveis = [nome for nome in dir(gx.expectations) if nome.startswith("Expect")]
print(f"{len(disponiveis)} expectativas prontas para usar. Uma amostra:\n")
for nome in disponiveis[:14]:
    print("  -", nome)


Great Expectations 1.22.0
59 expectativas prontas para usar. Uma amostra:

  - ExpectColumnDistinctValuesToBeInSet
  - ExpectColumnDistinctValuesToContainSet
  - ExpectColumnDistinctValuesToEqualSet
  - ExpectColumnKLDivergenceToBeLessThan
  - ExpectColumnMaxToBeBetween
  - ExpectColumnMeanToBeBetween
  - ExpectColumnMedianToBeBetween
  - ExpectColumnMinToBeBetween
  - ExpectColumnMostCommonValueToBeInSet
  - ExpectColumnPairValuesAToBeGreaterThanB
  - ExpectColumnPairValuesToBeEqual
  - ExpectColumnPairValuesToBeInSet
  - ExpectColumnProportionOfNonNullValuesToBeBetween
  - ExpectColumnProportionOfUniqueValuesToBeBetween


## 3. Montando a primeira validação, peça por peça

Vamos fazer o caminho completo na mão uma vez. Depois usamos o atalho do
projeto.

In [6]:
from f1_pipeline.quality import get_context

# 1) O contexto: o "projeto" de qualidade, persistido em gx/
contexto = get_context()

# 2) A fonte e 3) o ativo: nossa fonte são DataFrames pandas
fonte = contexto.data_sources.add_or_update_pandas("demo_pandas")
ativo = fonte.add_dataframe_asset(name="demo_results")

# 4) O lote: aqui, a tabela inteira de uma vez
lote = ativo.add_batch_definition_whole_dataframe("lote_completo")
print("Lote definido:", lote.name)


22:04:31 | INFO    | f1_pipeline.quality.context  | Contexto GX pronto em C:\dev\Estudo-MDS\gx
Lote definido: lote_completo


In [7]:
from great_expectations import expectations as gxe

# 5) A suite: o conjunto de regras
suite_demo = contexto.suites.add_or_update(gx.ExpectationSuite(name="demo_bronze_results"))

suite_demo.add_expectation(
    gxe.ExpectColumnValuesToNotBeNull(
        column="driver_id",
        meta={"descricao": "Sem piloto não existe resultado."},
    )
)
suite_demo.add_expectation(
    gxe.ExpectCompoundColumnsToBeUnique(
        column_list=["season", "round", "driver_id"],
        meta={"descricao": "Um piloto aparece uma única vez por corrida."},
    )
)
suite_demo.add_expectation(
    gxe.ExpectColumnValuesToMatchRegex(
        column="points",
        regex=r"^-?\d+(\.\d+)?$",
        meta={"descricao": "Pontos são numéricos, mesmo ainda em formato texto."},
    )
)

print(f"{len(suite_demo.expectations)} regras na suite '{suite_demo.name}'")


3 regras na suite 'demo_bronze_results'


### O campo `meta` é o que separa uma validação útil de um log técnico

Compare as duas mensagens de erro:

* `expect_compound_columns_to_be_unique falhou em [season, round, driver_id]`
* **"Um piloto aparece uma única vez por corrida"** — falhou em 12 linhas

A primeira só o engenheiro entende. A segunda vai direto para o time de negócio.
Por isso **toda** expectativa deste projeto carrega uma `descricao` em
português — e é ela que aparece no app Streamlit.

In [8]:
# 6) e 7) Validation Definition + Checkpoint, e a execução
definicao = contexto.validation_definitions.add_or_update(
    gx.ValidationDefinition(name="demo_vd", data=lote, suite=suite_demo)
)
checkpoint = contexto.checkpoints.add_or_update(
    gx.Checkpoint(name="demo_cp", validation_definitions=[definicao], result_format="SUMMARY")
)

resultado = checkpoint.run(batch_parameters={"dataframe": resultados_bronze})
print("Validação passou?", resultado.success)


Validação passou? True


In [9]:
# Lendo o resultado regra a regra
validacao = next(iter(resultado.run_results.values()))
for item in validacao.results:
    dados = item.to_json_dict()
    regra = dados["expectation_config"]["type"]
    inesperadas = dados["result"].get("unexpected_count", 0)
    icone = "OK  " if dados["success"] else "FALHA"
    print(f"{icone} {regra:<45} linhas fora do esperado: {inesperadas}")


OK   expect_column_values_to_not_be_null           linhas fora do esperado: 0
OK   expect_compound_columns_to_be_unique          linhas fora do esperado: 0
OK   expect_column_values_to_match_regex           linhas fora do esperado: 0


## 4. O atalho do projeto

Fazer isso à mão para 10 tabelas seria repetitivo. O projeto embrulha tudo em
uma função — `validate_dataframe` — que devolve um resumo já legível:

```python
resultado = validate_dataframe(df, suite, asset_name="bronze_results")
```

E as regras de cada tabela ficam declaradas em
`src/f1_pipeline/quality/suites.py`, separadas por camada. **Essa separação é
uma decisão de arquitetura**, não organização de arquivo:

* **suites de bronze validam o CONTRATO da fonte** — como tudo ainda é texto,
  testamos *forma*: as colunas existem? o campo numérico casa com `^\d+$`?
  a chave de negócio é única? o volume faz sentido?
* **suites de silver/gold validam a SEMÂNTICA** — já com tipos, testamos
  *significado*: pontuação entre 0 e 26, pódios ≥ vitórias, percentual em 0–100.

In [10]:
from f1_pipeline.quality import suites

for camada, registro in suites.ALL_SUITES.items():
    print(f"\n{camada.upper()}")
    for nome, fabrica in registro.items():
        suite = fabrica()
        print(f"  {nome:<26} {len(suite.expectations)} regras")



BRONZE
  races                      8 regras
  results                    10 regras
  qualifying                 6 regras
  pit_stops                  4 regras
  driver_standings           4 regras
  constructor_standings      3 regras

SILVER
  fact_result                7 regras
  fact_qualifying            3 regras

GOLD
  agg_driver_season          7 regras
  agg_constructor_season     3 regras


In [11]:
# As regras da tabela mais importante do projeto, em português:
for expectativa in suites.bronze_results_suite().expectations:
    alvo = expectativa.configuration.kwargs.get("column") or expectativa.configuration.kwargs.get("column_list", "(tabela)")
    print(f"- [{expectativa.configuration.type}] {alvo}")
    print(f"    {expectativa.meta.get('descricao', '')}\n")


- [expect_table_columns_to_match_set] (tabela)
    O contrato de colunas da fonte não mudou.

- [expect_table_row_count_to_be_between] (tabela)
    No máximo ~25 pilotos por GP em até 30 GPs por temporada.

- [expect_compound_columns_to_be_unique] ['season', 'round', 'driver_id']
    Um piloto aparece uma única vez por corrida — se duplicar, houve reprocessamento indevido da ingestão.

- [expect_column_values_to_not_be_null] driver_id
    Sem piloto não existe resultado.

- [expect_column_values_to_not_be_null] constructor_id
    Todo piloto corre por uma equipe.

- [expect_column_values_to_match_regex] points
    Pontos são numéricos (podem ter meio ponto).

- [expect_column_values_to_match_regex] grid
    Posição de largada é inteira (0 = largou do pit).

- [expect_column_values_to_match_regex] position
    Posição de chegada é inteira.

- [expect_column_values_to_not_be_null] status
    O status ('Finished', '+1 Lap', 'Engine'...) explica o resultado e é obrigatório.

- [expect_colu

## 5. Validando toda a camada bronze

In [12]:
from f1_pipeline.quality import results_to_dataframe, validate_dataframe

resultados_qualidade = []
for nome, fabrica in suites.BRONZE_SUITES.items():
    resultados_qualidade.append(
        validate_dataframe(
            bronze[nome],
            fabrica(),
            asset_name=f"bronze_{nome}",
            context=contexto,
            build_docs=False,
        )
    )

results_to_dataframe(resultados_qualidade)


22:05:02 | INFO    | f1_pipeline.quality.context  | Validando 'bronze_races' (90 linhas) com a suite 'bronze_races'
22:05:02 | INFO    | f1_pipeline.quality.context  | APROVADO | bronze_races: 8/8 regras aprovadas
22:05:03 | INFO    | f1_pipeline.quality.context  | Validando 'bronze_results' (1799 linhas) com a suite 'bronze_results'
22:05:03 | INFO    | f1_pipeline.quality.context  | APROVADO | bronze_results: 10/10 regras aprovadas
22:05:03 | INFO    | f1_pipeline.quality.context  | Validando 'bronze_qualifying' (1798 linhas) com a suite 'bronze_qualifying'
22:05:03 | INFO    | f1_pipeline.quality.context  | APROVADO | bronze_qualifying: 6/6 regras aprovadas
22:05:03 | INFO    | f1_pipeline.quality.context  | Validando 'bronze_pit_stops' (3340 linhas) com a suite 'bronze_pit_stops'
22:05:03 | INFO    | f1_pipeline.quality.context  | APROVADO | bronze_pit_stops: 4/4 regras aprovadas
22:05:03 | INFO    | f1_pipeline.quality.context  | Validando 'bronze_driver_standings' (89 linhas) com

,tabela,regras,aprovadas,reprovadas,taxa_sucesso_%,linhas,status
0,bronze_races,8,8,0,100.0,90,APROVADO
1,bronze_results,10,10,0,100.0,1799,APROVADO
2,bronze_qualifying,6,6,0,100.0,1798,APROVADO
3,bronze_pit_stops,4,4,0,100.0,3340,APROVADO
4,bronze_driver_standings,4,4,0,100.0,89,APROVADO
5,bronze_constructor_standings,3,3,0,100.0,40,APROVADO


## 6. E quando o dado está errado? Vamos quebrar de propósito

Uma validação que nunca reprova não prova nada. Para confiar no portão,
precisamos vê-lo fechar.

Vamos introduzir três defeitos clássicos de ingestão em uma cópia do bronze:

1. **duplicação** — a mesma linha gravada duas vezes (reprocessamento mal feito);
2. **valor ausente** — um `driver_id` nulo (campo que sumiu na origem);
3. **valor impossível** — `points = "muitos"` (mudança de tipo na fonte).

In [13]:
dado_corrompido = bronze["results"].copy()

# defeito 1: duplica a primeira linha
dado_corrompido = pd.concat([dado_corrompido, dado_corrompido.head(1)], ignore_index=True)

# defeito 2: apaga um identificador obrigatório
dado_corrompido.loc[5, "driver_id"] = None

# defeito 3: um valor que não é numérico onde deveria ser
dado_corrompido.loc[7, "points"] = "muitos"

print(f"{len(bronze['results'])} linhas originais -> {len(dado_corrompido)} linhas corrompidas")
dado_corrompido.loc[[5, 7], ["season", "round", "driver_id", "points"]]


1799 linhas originais -> 1800 linhas corrompidas


,season,round,driver_id,points
5,2021,1,None,8
7,2021,1,sainz,muitos


In [14]:
resultado_ruim = validate_dataframe(
    dado_corrompido,
    suites.bronze_results_suite(),
    asset_name="bronze_results_corrompido",
    context=contexto,
    build_docs=False,
)

print(f"\nStatus: {'APROVADO' if resultado_ruim.sucesso else 'REPROVADO'}")
print(f"{resultado_ruim.regras_falhas} de {resultado_ruim.total_regras} regras falharam\n")

for falha in resultado_ruim.failures():
    print(f"[X] {falha['regra']}  ->  coluna: {falha['coluna']}")
    print(f"    {falha['descricao']}")
    print(f"    {falha['linhas_inesperadas']} linha(s) fora do esperado "
          f"({falha['percentual_inesperado']}%)")
    if falha["exemplos"]:
        print(f"    exemplos: {falha['exemplos']}")
    print()


22:05:20 | INFO    | f1_pipeline.quality.context  | Validando 'bronze_results_corrompido' (1800 linhas) com a suite 'bronze_results'
22:05:20 | ERROR   | f1_pipeline.quality.context  | REPROVADO | bronze_results_corrompido: 7/10 regras aprovadas

Status: REPROVADO
3 de 10 regras falharam

[X] expect_compound_columns_to_be_unique  ->  coluna: season, round, driver_id
    Um piloto aparece uma única vez por corrida — se duplicar, houve reprocessamento indevido da ingestão.
    2 linha(s) fora do esperado (0.111%)
    exemplos: ["{'season': '2021', 'round': '1', 'driver_id': 'hamilton'}", "{'season': '2021', 'round': '1', 'driver_id': 'hamilton'}"]

[X] expect_column_values_to_not_be_null  ->  coluna: driver_id
    Sem piloto não existe resultado.
    1 linha(s) fora do esperado (0.056%)
    exemplos: ['None']

[X] expect_column_values_to_match_regex  ->  coluna: points
    Pontos são numéricos (podem ter meio ponto).
    1 linha(s) fora do esperado (0.056%)
    exemplos: ['muitos']



### Leia o que aconteceu

As três regras certas reprovaram, cada uma apontando **a linha exata** e **o
valor exato** que causou o problema. É essa precisão que transforma "os dados
estão estranhos" em um chamado que alguém consegue resolver.

> **Lição.** Escreva pelo menos uma expectativa para cada suposição que o seu
> código faz. Se o seu `groupby` assume que a chave é única, existe uma
> expectativa de unicidade esperando para ser escrita.

## 7. `mostly`: tolerância explícita

Nem toda regra é absoluta. Nem todo piloto marca tempo no Q1 — quem quebra no
aquecimento não marca. Uma regra de "não nulo" reprovaria a temporada inteira por
causa de dois casos legítimos.

O GX resolve isso com `mostly`:

```python
gxe.ExpectColumnValuesToNotBeNull(column="q1", mostly=0.95)
# "aceito até 5% de ausências; acima disso, algo está errado"
```

Isso é diferente de ignorar o problema: a tolerância fica **escrita, versionada e
justificada**, em vez de existir só na cabeça de quem escreveu o pipeline.

In [15]:
quali = bronze["qualifying"]
faltantes = quali["q1"].isna().mean() * 100
print(f"Pilotos sem tempo em Q1: {faltantes:.2f}%  (a regra tolera até 5%)")
print(f"Sem tempo em Q2:         {quali['q2'].isna().mean() * 100:.1f}%  <- normal: só 15 passam")
print(f"Sem tempo em Q3:         {quali['q3'].isna().mean() * 100:.1f}%  <- normal: só 10 passam")


Pilotos sem tempo em Q1: 0.00%  (a regra tolera até 5%)
Sem tempo em Q2:         25.4%  <- normal: só 15 passam
Sem tempo em Q3:         50.6%  <- normal: só 10 passam


## 8. Data Docs: o relatório que outras pessoas leem

O GX gera um site HTML com o histórico de todas as validações. É o artefato que
você manda para o time de negócio e para a auditoria.

O projeto copia esse site para `docs/data_docs/` — pasta versionável, que pode
ser publicada no GitHub Pages.

In [16]:
from f1_pipeline.quality import publish_data_docs, save_quality_report

caminho_docs = publish_data_docs(contexto)
caminho_relatorio = save_quality_report(resultados_qualidade, etapa="validacao_bronze")

print("\nRelatório HTML :", caminho_docs)
print("Relatório JSON :", caminho_relatorio)
print("\nAbra o HTML no navegador — e note que o JSON é o mesmo arquivo que")
print("alimenta a página 'Qualidade dos Dados' do app Streamlit.")


22:05:38 | INFO    | f1_pipeline.quality.context  | Data Docs publicados em C:\dev\Estudo-MDS\docs\data_docs\index.html
22:05:38 | INFO    | f1_pipeline.quality.context  | Relatório de qualidade salvo em C:\dev\Estudo-MDS\reports\quality_report.json

Relatório HTML : C:\dev\Estudo-MDS\docs\data_docs\index.html
Relatório JSON : C:\dev\Estudo-MDS\reports\quality_report.json

Abra o HTML no navegador — e note que o JSON é o mesmo arquivo que
alimenta a página 'Qualidade dos Dados' do app Streamlit.


## 9. O portão de qualidade no pipeline

Em produção, a validação não é informativa: ela **bloqueia**. É o argumento
`raise_on_failure=True`.

```python
validate_dataframe(df, suite, asset_name="bronze_results", raise_on_failure=True)
# -> ValueError: Qualidade reprovada em 'bronze_results': ...
```

No orquestrador (`scripts/run_pipeline.py`) isso está exposto como
`--parar-se-reprovar`. A sequência real do pipeline é:

```
ingestão → bronze → [PORTÃO 1] → silver → [PORTÃO 2] → gold → [PORTÃO 3]
```

Cada portão pergunta uma coisa diferente:

1. **bronze** — a fonte continua cumprindo o contrato?
2. **silver** — a transformação preservou a verdade do dado?
3. **gold** — a agregação reconcilia com o detalhe?

In [17]:
try:
    validate_dataframe(
        dado_corrompido,
        suites.bronze_results_suite(),
        asset_name="bronze_results_portao",
        context=contexto,
        build_docs=False,
        raise_on_failure=True,   # <- comportamento de produção
    )
except ValueError as erro:
    print("O pipeline PAROU, como deveria:\n")
    print(erro)


22:05:42 | INFO    | f1_pipeline.quality.context  | Validando 'bronze_results_portao' (1800 linhas) com a suite 'bronze_results'
22:05:42 | ERROR   | f1_pipeline.quality.context  | REPROVADO | bronze_results_portao: 7/10 regras aprovadas
O pipeline PAROU, como deveria:

Qualidade reprovada em 'bronze_results_portao': expect_compound_columns_to_be_unique(season, round, driver_id), expect_column_values_to_not_be_null(driver_id), expect_column_values_to_match_regex(points)


## 10. O que fica desta etapa

* **Dado ruim não avisa.** Ele passa, roda e produz número errado. Validação
  explícita é a única defesa.
* **Valide cedo e valide entre camadas.** Um portão só no fim descobre o
  problema tarde demais.
* **Escreva a regra em português.** `expect_column_values_to_not_be_null` é para
  a máquina; "sem piloto não existe resultado" é para as pessoas.
* **Toda suposição do seu código merece uma expectativa.**
* **Tolerância se declara** com `mostly`, não se ignora em silêncio.
* **Bronze valida forma; silver e gold validam significado.**

### Exercícios

1. Escreva uma expectativa garantindo que `date` em `bronze_races` está dentro
   da temporada correspondente (dica: `ExpectColumnValuesToMatchRegex` com o ano).
2. A regra de unicidade de `bronze_pit_stops` usa `[season, round, driver_id, stop]`.
   O que acontece se você tirar `stop` da lista? Teste e explique o resultado.
3. Crie uma suite para `bronze_qualifying` que garanta que o tempo de Q3, quando
   existe, é menor que o de Q1 do mesmo piloto (dica:
   `ExpectColumnPairValuesAToBeGreaterThanB`) — e explique por que essa regra
   precisa da camada **silver** para funcionar.

---

**Próximo:** [`03_processamento_agregacoes.ipynb`](03_processamento_agregacoes.ipynb) —
transformar dado validado em resposta.